# Week 3 Practice Lab: Market Analysis in Action

**BUS 696: Generative AI in Finance**  
**Professor Jonathan Hersh**

---

## Learning Objectives

By the end of this lab, you will be able to:
1. Conduct an **event study** around a major news event (Trump tariff reversal)
2. Measure **abnormal returns** — did a stock move more than the market expected?
3. Understand how **puts and calls** can be used to trade news events (hedging, directional bets)
4. Compute and interpret the **RSI** (Relative Strength Index) for tech vs. consumer durables stocks
5. Compare **PE ratios** across sectors and understand the SaaS valuation decline vs. "boring stocks" rally
6. Synthesize concepts from Weeks 1–3: returns, risk, CAPM, technical indicators, and market efficiency
7. Use **prompt engineering** to generate AI-assisted financial analysis

---

**Note:** This is a practice lab — work through it at your own pace. The goal is to reinforce what you've learned in Weeks 1–3 through a real-world, timely application.

## Part 1: Setup

In [ ]:
# Install if needed (run once, then comment out)
# !pip install yfinance pandas matplotlib plotly scipy scikit-learn

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Color palette
NAVY = '#1E2761'
CORAL = '#F96167'
TEAL = '#028090'
GOLD = '#F9A825'
GRAY = '#6c757d'
GREEN = '#2E7D32'

print('All libraries loaded!')

---

## Part 2: Trading the News — The Trump Tariff Reversal

One of the most dramatic market events of early 2025 was the **Trump tariff saga**:

- **February 1, 2025**: President Trump announced broad tariffs — 25% on Canada and Mexico, 10% on China
- **February 3, 2025 (Monday)**: Markets opened sharply lower as investors priced in the tariff impact
- **February 3–4, 2025**: After negotiations, Trump announced a **90-day pause** on tariffs for Canada and Mexico
- **Markets reversed hard** — stocks that had plunged on the tariff news surged on the reversal

This is a textbook **event study**: a discrete, identifiable event that moves markets. Let's analyze what happened, which stocks were affected, and whether the reversal created a trading opportunity.

### Why This Matters for AI in Finance

News-driven trading is one of the areas where AI/NLP models have shown the most promise:
- **Sentiment analysis** of headlines can detect market-moving events in milliseconds
- **Event studies** are a core tool for measuring whether markets react efficiently
- Understanding **how quickly** prices adjust tells us about market efficiency (EMH)

In [ ]:
# ============================================================
# DOWNLOAD DATA: Stocks most affected by tariffs
# ============================================================

# Tariff-sensitive stocks (heavy trade exposure to Mexico/Canada/China)
tariff_sensitive = ['GM', 'F', 'CAT', 'DE', 'AVGO']  # Autos, industrials, semiconductors

# Domestic-focused stocks (less tariff exposure)
domestic_focused = ['UNH', 'VZ', 'DUK', 'WM', 'ADP']  # Healthcare, utilities, services

# Market benchmark
benchmark = ['SPY']

all_tickers = tariff_sensitive + domestic_focused + benchmark

# Download data around the event (Jan 2025 – Feb 2025)
data = yf.download(all_tickers, start='2025-01-02', end='2025-02-28')
close = data['Close']

# Calculate daily returns
returns = close.pct_change()

print(f'Downloaded {len(close)} trading days for {len(all_tickers)} tickers')
print(f'Date range: {close.index[0].date()} to {close.index[-1].date()}')
print(f'\nTariff-sensitive: {tariff_sensitive}')
print(f'Domestic-focused: {domestic_focused}')

### 2a. What Happened on the Event Days?

Let's zoom into the key dates:
- **Jan 31 (Friday)**: Tariff announcement after hours
- **Feb 3 (Monday)**: Market reaction — the "tariff shock"
- **Feb 3–4**: Tariff pause announced — the "reversal"

In [ ]:
# ============================================================
# EVENT DAY RETURNS
# ============================================================

# Key event dates
event_dates = ['2025-02-03', '2025-02-04', '2025-02-05']

print('Returns on Key Event Days')
print('=' * 80)

for date in event_dates:
    try:
        date_ts = pd.Timestamp(date)
        if date_ts in returns.index:
            day_returns = returns.loc[date_ts]
            print(f'\n{date} ({date_ts.strftime("%A")}):')
            print(f'  SPY (Market):  {day_returns["SPY"]:+.2%}')
            print(f'  --- Tariff-Sensitive ---')
            for t in tariff_sensitive:
                print(f'  {t:6s}: {day_returns[t]:+.2%}')
            print(f'  --- Domestic-Focused ---')
            for t in domestic_focused:
                print(f'  {t:6s}: {day_returns[t]:+.2%}')
    except:
        print(f'\n{date}: No trading data (may be a weekend/holiday)')

### 2b. Visualizing the Tariff Shock and Reversal

Let's normalize prices so we can compare all stocks on the same chart — starting from 100 on January 31 (the day before the tariff announcement).

In [ ]:
# ============================================================
# NORMALIZED PRICE CHART: Tariff-Sensitive vs. Domestic
# ============================================================

# Normalize to Jan 31 = 100 (last trading day before event)
base_date = pd.Timestamp('2025-01-31')
# Find the closest trading day on or before base_date
available = close.index[close.index <= base_date]
if len(available) > 0:
    base_date = available[-1]

normalized = (close / close.loc[base_date]) * 100

# Create equal-weighted group baskets
tariff_basket = normalized[tariff_sensitive].mean(axis=1)
domestic_basket = normalized[domestic_focused].mean(axis=1)
spy_norm = normalized['SPY']

fig, ax = plt.subplots(figsize=(14, 7))

ax.plot(tariff_basket.index, tariff_basket, color=CORAL, linewidth=2.5,
        label=f'Tariff-Sensitive ({tariff_basket.iloc[-1]:.1f})')
ax.plot(domestic_basket.index, domestic_basket, color=TEAL, linewidth=2.5,
        label=f'Domestic-Focused ({domestic_basket.iloc[-1]:.1f})')
ax.plot(spy_norm.index, spy_norm, color=GRAY, linewidth=2, linestyle='--',
        label=f'SPY Benchmark ({spy_norm.iloc[-1]:.1f})')

# Mark the event dates
for date_str, label, ypos in [('2025-02-03', 'Tariff\nShock', 96),
                                ('2025-02-04', 'Tariff\nPause', 103)]:
    date_ts = pd.Timestamp(date_str)
    if date_ts in spy_norm.index:
        ax.axvline(x=date_ts, color='black', linewidth=1, linestyle=':', alpha=0.5)
        ax.annotate(label, xy=(date_ts, ypos), fontsize=10, fontweight='bold',
                    ha='center', va='bottom',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7))

ax.axhline(y=100, color='black', linewidth=0.5, linestyle=':')
ax.set_xlabel('Date', fontsize=13)
ax.set_ylabel('Normalized Price (100 = Jan 31)', fontsize=13)
ax.set_title('The Trump Tariff Shock and Reversal\n'
             'Tariff-sensitive stocks dropped and then recovered; domestic stocks barely flinched',
             fontweight='bold', fontsize=14)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print('Key observation: The tariff-sensitive stocks moved much more than the market.')
print('The domestic-focused stocks were relatively unaffected.')
print('This is what we\'d expect if markets correctly price tariff exposure.')

### 2c. Measuring Abnormal Returns

An **abnormal return** is the return *beyond* what we'd expect given how the overall market moved. From Week 2, recall that CAPM gives us:

$$R_{\text{stock}} = \alpha + \beta \times R_{\text{market}} + \epsilon$$

The **abnormal return** on a given day is:

$$AR_t = R_{\text{stock},t} - (\hat{\alpha} + \hat{\beta} \times R_{\text{market},t})$$

In other words: how much did the stock move *beyond* what its beta predicted?

Let's compute this for the tariff event days.

In [ ]:
# ============================================================
# ABNORMAL RETURNS: Using CAPM regression
# ============================================================

# Step 1: Estimate beta using pre-event data (estimation window)
# Use returns from before the event to fit the model
pre_event = returns.loc[:'2025-01-30'].dropna()

# Step 2: Compute abnormal returns for each stock on event days
event_window = returns.loc['2025-02-03':'2025-02-07'].dropna()

print('Abnormal Returns (Event Window: Feb 3–7, 2025)')
print('=' * 75)
print(f'{"Stock":>6s}   {"Beta":>6s}  {"Feb 3":>8s}  {"Feb 4":>8s}  {"Feb 5":>8s}  {"Cumul. AR":>10s}')
print('-' * 75)

for group_name, tickers in [('TARIFF-SENSITIVE', tariff_sensitive), 
                              ('DOMESTIC-FOCUSED', domestic_focused)]:
    print(f'\n  --- {group_name} ---')
    for ticker in tickers:
        # Estimate beta from pre-event window
        stock_ret = pre_event[ticker].dropna()
        mkt_ret = pre_event['SPY'].loc[stock_ret.index]
        if len(stock_ret) < 10:
            continue
        slope, intercept, _, _, _ = stats.linregress(mkt_ret, stock_ret)
        
        # Compute abnormal returns in event window
        abnormal_returns = []
        for date in event_window.index[:3]:  # First 3 event days
            if ticker in returns.columns and date in returns.index:
                actual = returns.loc[date, ticker]
                expected = intercept + slope * returns.loc[date, 'SPY']
                ar = actual - expected
                abnormal_returns.append(ar)
            else:
                abnormal_returns.append(np.nan)
        
        cumulative_ar = np.nansum(abnormal_returns)
        ar_strs = [f'{ar:+.2%}' if not np.isnan(ar) else '   N/A' for ar in abnormal_returns]
        print(f'{ticker:>6s}   {slope:>6.2f}  {ar_strs[0]:>8s}  {ar_strs[1]:>8s}  {ar_strs[2]:>8s}  {cumulative_ar:>+9.2%}')

print('\n\nPositive AR = stock did better than its beta predicted (good news for that stock)')
print('Negative AR = stock did worse than its beta predicted (bad news for that stock)')

### 2d. The Speed of Market Adjustment

A key question for market efficiency: **how quickly did prices adjust?**

If markets are efficient, the adjustment should be nearly instantaneous — you shouldn't be able to profit by trading *after* the news is public.

Let's look at cumulative abnormal returns (CARs) over the event window to see if there was any "drift" (slow adjustment) or if it was all priced in immediately.

In [ ]:
# ============================================================
# CUMULATIVE ABNORMAL RETURNS (CARs)
# ============================================================

# Compute CARs for the tariff-sensitive basket vs domestic basket
# Use a wider window: 5 days before to 10 days after
full_window = returns.loc['2025-01-24':'2025-02-14'].dropna()

fig, ax = plt.subplots(figsize=(14, 7))

for group_name, tickers, color in [('Tariff-Sensitive', tariff_sensitive, CORAL),
                                     ('Domestic-Focused', domestic_focused, TEAL)]:
    group_cars = []
    for ticker in tickers:
        stock_ret = pre_event[ticker].dropna()
        mkt_ret = pre_event['SPY'].loc[stock_ret.index]
        if len(stock_ret) < 10:
            continue
        slope, intercept, _, _, _ = stats.linregress(mkt_ret, stock_ret)
        
        # Abnormal returns over full window
        ars = []
        for date in full_window.index:
            actual = returns.loc[date, ticker]
            expected = intercept + slope * returns.loc[date, 'SPY']
            ars.append(actual - expected)
        group_cars.append(ars)
    
    # Average CARs across stocks in the group
    avg_cars = np.cumsum(np.nanmean(group_cars, axis=0)) * 100
    ax.plot(full_window.index, avg_cars, color=color, linewidth=2.5,
            label=f'{group_name} (CAR = {avg_cars[-1]:+.1f}%)')

# Mark event
event_ts = pd.Timestamp('2025-02-03')
if event_ts in full_window.index:
    ax.axvline(x=event_ts, color='black', linewidth=2, linestyle='--', alpha=0.7)
    ax.annotate('Tariff\nAnnouncement', xy=(event_ts, ax.get_ylim()[1] * 0.8),
                fontsize=11, fontweight='bold', ha='center',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7))

ax.axhline(y=0, color='black', linewidth=0.5)
ax.set_xlabel('Date', fontsize=13)
ax.set_ylabel('Cumulative Abnormal Return (%)', fontsize=13)
ax.set_title('Cumulative Abnormal Returns Around the Tariff Event\n'
             'How quickly did the market process the news?',
             fontweight='bold', fontsize=14)
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

print('If markets are efficient, most of the price adjustment happens immediately.')
print('Any post-announcement drift would suggest the market was slow to react.')
print('Look at the chart: did prices keep moving after the event, or did they stabilize?')

**Pause and reflect:**
- Did the tariff-sensitive stocks show a clear negative reaction on Feb 3?
- How quickly did the reversal happen when the tariff pause was announced?
- Could you have profited by trading *after* the news was public? (What does this say about EMH?)

---

### 2e. Trading the News with Options — Puts, Calls, and the Tariff Event

In Week 2, you learned about **options** — contracts that give you the right (not obligation) to buy or sell a stock at a specific price. Let's see how a trader could have used options around the tariff event.

**Quick recap:**
- **Put option**: Right to **sell** at the strike price → profits when stock **falls** (downside bet / insurance)
- **Call option**: Right to **buy** at the strike price → profits when stock **rises** (upside bet)

### Why options for news events?

Options are powerful tools for trading around anticipated events because:
1. **Leverage**: A small premium controls a large position (amplifies gains *and* losses)
2. **Defined risk**: Max loss = the premium you paid (unlike shorting, where losses are theoretically unlimited)
3. **Asymmetry**: You can bet on a big move in one direction while capping your downside

### Three option strategies a trader might have used around the tariff announcement:

In [ ]:
# ============================================================
# OPTION STRATEGIES FOR THE TARIFF EVENT
# ============================================================

# Let's use GM (General Motors) as our example — heavily exposed to Mexico/Canada tariffs.
# We'll look at the actual GM prices around the tariff event.

gm_pre_event_price = close['GM'].loc[:pd.Timestamp('2025-01-31')].iloc[-1]
gm_post_shock = close['GM'].loc[pd.Timestamp('2025-02-03'):].iloc[0]  # Feb 3 close
gm_post_reversal = close['GM'].loc[pd.Timestamp('2025-02-04'):].iloc[0]  # Feb 4 close

print(f'GM Price Timeline:')
print(f'  Jan 31 (pre-event):    ${gm_pre_event_price:.2f}')
print(f'  Feb 3 (tariff shock):  ${gm_post_shock:.2f} ({(gm_post_shock/gm_pre_event_price - 1)*100:+.1f}%)')
print(f'  Feb 4 (tariff pause):  ${gm_post_reversal:.2f} ({(gm_post_reversal/gm_pre_event_price - 1)*100:+.1f}%)')

# ============================================================
# Three option strategies around the tariff event
# ============================================================

strike_put = round(gm_pre_event_price * 0.98)  # ~2% out-of-the-money put
put_premium = 1.50

# Range of possible GM prices
prices_range = np.linspace(gm_pre_event_price * 0.85, gm_pre_event_price * 1.15, 200)

# Stock P&L (from owning shares)
stock_pnl = prices_range - gm_pre_event_price
put_pnl = np.maximum(strike_put - prices_range, 0) - put_premium
protective_put_pnl = stock_pnl + put_pnl

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# --- Strategy 1: Protective Put (insurance against crash) ---
ax = axes[0]
ax.plot(prices_range, stock_pnl, '--', color=NAVY, linewidth=1.5, alpha=0.6, label='Stock only')
ax.plot(prices_range, put_pnl, '--', color=CORAL, linewidth=1.5, alpha=0.6, label='Put only')
ax.plot(prices_range, protective_put_pnl, color='black', linewidth=2.5, label='Protective put')
ax.axhline(y=0, color='gray', linewidth=0.5)
ax.axvline(x=gm_pre_event_price, color='gray', linewidth=0.5, linestyle=':')
ax.fill_between(prices_range, protective_put_pnl, 0,
                where=(protective_put_pnl > 0), alpha=0.1, color='green')
ax.fill_between(prices_range, protective_put_pnl, 0,
                where=(protective_put_pnl < 0), alpha=0.1, color='red')
# Mark Feb 3 crash price
ax.axvline(x=gm_post_shock, color=CORAL, linewidth=2, linestyle='--', alpha=0.7)
ax.annotate(f'Feb 3\n${gm_post_shock:.0f}', xy=(gm_post_shock, -5),
            fontsize=9, fontweight='bold', color=CORAL, ha='center')
ax.set_xlabel('GM Price at Expiration ($)')
ax.set_ylabel('Profit / Loss ($)')
ax.set_title('Strategy 1: Protective Put\n"Insurance" — own stock + buy put',
             fontweight='bold', fontsize=12)
ax.legend(fontsize=9)

# --- Strategy 2: Buy Puts (bearish bet on tariffs) ---
ax = axes[1]
ax.plot(prices_range, put_pnl, color=CORAL, linewidth=2.5, label=f'Buy Put (strike=${strike_put})')
ax.axhline(y=0, color='gray', linewidth=0.5)
ax.fill_between(prices_range, put_pnl, 0,
                where=(put_pnl > 0), alpha=0.15, color='green')
ax.fill_between(prices_range, put_pnl, 0,
                where=(put_pnl < 0), alpha=0.15, color='red')
ax.axvline(x=gm_post_shock, color=CORAL, linewidth=2, linestyle='--', alpha=0.7)
# Show actual P&L at the Feb 3 price
pnl_at_shock = max(strike_put - gm_post_shock, 0) - put_premium
ax.scatter([gm_post_shock], [pnl_at_shock], s=120, color=CORAL, edgecolor='black', zorder=5)
ax.annotate(f'P&L: ${pnl_at_shock:+.2f}/share', xy=(gm_post_shock, pnl_at_shock),
            textcoords='offset points', xytext=(15, 10), fontsize=10, fontweight='bold')
ax.set_xlabel('GM Price at Expiration ($)')
ax.set_ylabel('Profit / Loss ($)')
ax.set_title('Strategy 2: Buy Puts\n"Tariffs will hurt GM"',
             fontweight='bold', fontsize=12)
ax.legend(fontsize=9)

# --- Strategy 3: Buy Calls on the dip (bet on reversal) ---
ax = axes[2]
strike_call = round(gm_post_shock)  # At-the-money call bought after the crash
call_premium = 2.00
call_pnl = np.maximum(prices_range - strike_call, 0) - call_premium

ax.plot(prices_range, call_pnl, color=TEAL, linewidth=2.5,
        label=f'Buy Call (strike=${strike_call})')
ax.axhline(y=0, color='gray', linewidth=0.5)
ax.fill_between(prices_range, call_pnl, 0,
                where=(call_pnl > 0), alpha=0.15, color='green')
ax.fill_between(prices_range, call_pnl, 0,
                where=(call_pnl < 0), alpha=0.15, color='red')
ax.axvline(x=gm_post_reversal, color=TEAL, linewidth=2, linestyle='--', alpha=0.7)
# Show actual P&L at the Feb 4 reversal price
pnl_at_reversal = max(gm_post_reversal - strike_call, 0) - call_premium
ax.scatter([gm_post_reversal], [pnl_at_reversal], s=120, color=TEAL, edgecolor='black', zorder=5)
ax.annotate(f'P&L: ${pnl_at_reversal:+.2f}/share', xy=(gm_post_reversal, pnl_at_reversal),
            textcoords='offset points', xytext=(15, 10), fontsize=10, fontweight='bold')
ax.set_xlabel('GM Price at Expiration ($)')
ax.set_ylabel('Profit / Loss ($)')
ax.set_title('Strategy 3: Buy Calls on the Dip\n"Tariffs will be reversed"',
             fontweight='bold', fontsize=12)
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

print('Three ways options could have been used around the tariff event:')
print(f'  1. PROTECTIVE PUT: Own GM + buy ${strike_put} put for ${put_premium}')
print(f'     → Limits downside if tariffs tank the stock, keeps upside if they don\'t')
print(f'     → Cost: ${put_premium} per share in premium (the price of insurance)')
print(f'  2. BUY PUTS (bearish): Bet GM will fall below ${strike_put - put_premium:.2f}')
print(f'     → Max loss = ${put_premium} (the premium). Max gain = ${strike_put - put_premium:.2f} (if GM → $0)')
print(f'  3. BUY CALLS ON THE DIP: After Feb 3 crash, bet on a reversal')
print(f'     → Max loss = ${call_premium} (the premium). Profits if GM rebounds above ${strike_call + call_premium:.2f}')

### Options Takeaways for Event-Driven Trading

**Key insight:** Options let you express a *view* on news events with **defined risk**.

| If you think... | Strategy | Max Loss | Potential Gain |
|----------------|----------|----------|----------------|
| "Tariffs will hurt GM" | Buy puts | Premium paid | Large if stock crashes |
| "I own GM and want insurance" | Protective put (stock + put) | Premium paid | Unlimited upside minus premium |
| "The crash is overdone, it'll reverse" | Buy calls after the dip | Premium paid | Large if stock rebounds |
| "Big move coming, unsure of direction" | Straddle (buy call + put) | Both premiums | Large if stock moves a lot either way |

**The EMH connection:** If markets are efficient, option prices *already reflect* the probability of tariff scenarios. The put premium you'd pay on Jan 31 would be high precisely because the market knows tariff risk exists. This is **implied volatility** — the market's estimate of how much a stock might move.

**The practical challenge:** You can't consistently buy "cheap" options before news events because the options market is pricing in the same information you have. This is the options market's version of market efficiency.

---

## Part 3: RSI Analysis — Tech vs. Consumer Durables

### What is RSI and Why Do Traders Use It?

The **Relative Strength Index (RSI)** is one of the most popular **momentum oscillators** in technical analysis. It was developed by J. Welles Wilder in 1978, and virtually every trading platform displays it.

RSI measures the **speed and magnitude of recent price changes** on a scale of 0 to 100. The intuition is simple:

| RSI Range | Name | What It Means | Trader Interpretation |
|-----------|------|---------------|----------------------|
| **> 70** | Overbought | Price has risen fast relative to recent history | Stock may be "due for a pullback" — too many buyers, exhaustion |
| **30–70** | Neutral | Normal trading range | No strong signal |
| **< 30** | Oversold | Price has fallen fast relative to recent history | Stock may be "due for a bounce" — selling overdone |

### How RSI is calculated (step by step):

1. **Daily price changes**: How much did the stock go up or down each day?
2. **Separate gains and losses**: Keep gains positive, losses positive (absolute value)
3. **Average gain / Average loss** over a 14-day rolling window = **RS** (Relative Strength)
4. **RSI = 100 - (100 / (1 + RS))**

When gains dominate → RS is high → RSI approaches 100 (overbought)  
When losses dominate → RS is low → RSI approaches 0 (oversold)

### The Big Question (connects to Week 3):

If markets are efficient, RSI shouldn't predict future returns — the information is already priced in. But traders use it anyway. Is RSI a useful signal, or just noise? Let's look at the data.

Let's compare RSI patterns for **tech/SaaS stocks** vs. **consumer durables/staples** — two sectors that have behaved very differently recently.

In [ ]:
# ============================================================
# DOWNLOAD DATA: Tech/SaaS vs. Consumer Durables
# ============================================================

# Tech / SaaS stocks (high-growth, high-multiple names)
tech_tickers = ['MSFT', 'NVDA', 'CRM', 'ADBE', 'NOW', 'AAPL']

# Consumer durables / staples ("boring" stocks)
durables_tickers = ['WMT', 'PG', 'KO', 'JNJ', 'HD', 'MCD']

all_sector_tickers = tech_tickers + durables_tickers

# Download 1 year of data
sector_data = yf.download(all_sector_tickers, period='1y')
sector_close = sector_data['Close']

print(f'Downloaded {len(sector_close)} trading days')
print(f'\nTech/SaaS: {tech_tickers}')
print(f'Consumer Durables/Staples: {durables_tickers}')

In [ ]:
# ============================================================
# RSI COMPUTATION
# ============================================================

def compute_rsi(prices, window=14):
    """Compute the Relative Strength Index from a price series."""
    delta = prices.diff()
    gain = delta.where(delta > 0, 0).rolling(window).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window).mean()
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

# Compute RSI for all stocks
rsi_data = pd.DataFrame()
for ticker in all_sector_tickers:
    rsi_data[ticker] = compute_rsi(sector_close[ticker])

# Show current RSI values
print('Current RSI Values (most recent trading day)')
print('=' * 50)
print(f'\n  Tech/SaaS:')
for t in tech_tickers:
    rsi_val = rsi_data[t].iloc[-1]
    status = 'OVERBOUGHT' if rsi_val > 70 else ('OVERSOLD' if rsi_val < 30 else 'neutral')
    print(f'    {t:6s}: RSI = {rsi_val:5.1f}  ({status})')

print(f'\n  Consumer Durables/Staples:')
for t in durables_tickers:
    rsi_val = rsi_data[t].iloc[-1]
    status = 'OVERBOUGHT' if rsi_val > 70 else ('OVERSOLD' if rsi_val < 30 else 'neutral')
    print(f'    {t:6s}: RSI = {rsi_val:5.1f}  ({status})')

In [ ]:
# ============================================================
# RSI VISUALIZATION: Sector Average with Range + Current Snapshot
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# --- Top Left: Sector Average RSI Over Time (with min/max range) ---
ax = axes[0, 0]
# Tech sector
tech_rsi_mean = rsi_data[tech_tickers].mean(axis=1)
tech_rsi_min = rsi_data[tech_tickers].min(axis=1)
tech_rsi_max = rsi_data[tech_tickers].max(axis=1)
ax.plot(tech_rsi_mean.index, tech_rsi_mean, color=NAVY, linewidth=2.5, label='Tech/SaaS (avg)')
ax.fill_between(tech_rsi_mean.index, tech_rsi_min, tech_rsi_max, color=NAVY, alpha=0.1)

# Durables sector
dur_rsi_mean = rsi_data[durables_tickers].mean(axis=1)
dur_rsi_min = rsi_data[durables_tickers].min(axis=1)
dur_rsi_max = rsi_data[durables_tickers].max(axis=1)
ax.plot(dur_rsi_mean.index, dur_rsi_mean, color=TEAL, linewidth=2.5, label='Durables (avg)')
ax.fill_between(dur_rsi_mean.index, dur_rsi_min, dur_rsi_max, color=TEAL, alpha=0.1)

ax.axhline(y=70, color=CORAL, linewidth=1.5, linestyle='--', alpha=0.5)
ax.axhline(y=30, color=GREEN, linewidth=1.5, linestyle='--', alpha=0.5)
ax.fill_between(tech_rsi_mean.index, 70, 85, alpha=0.05, color=CORAL)
ax.fill_between(tech_rsi_mean.index, 15, 30, alpha=0.05, color=GREEN)
ax.set_ylim(15, 85)
ax.set_ylabel('RSI', fontsize=12)
ax.set_title('Sector Average RSI Over Time\n(shaded = range across stocks in sector)', fontweight='bold')
ax.legend(fontsize=11)

# --- Top Right: Current RSI Snapshot (bar chart) ---
ax = axes[0, 1]
current_rsi = rsi_data.iloc[-1][all_sector_tickers].sort_values(ascending=True)
colors_bar = []
for t in current_rsi.index:
    rsi_val = current_rsi[t]
    if rsi_val > 70:
        colors_bar.append(CORAL)   # Overbought
    elif rsi_val < 30:
        colors_bar.append(GREEN)   # Oversold
    elif t in tech_tickers:
        colors_bar.append(NAVY)    # Tech, neutral
    else:
        colors_bar.append(TEAL)    # Durables, neutral

bars = ax.barh(current_rsi.index, current_rsi.values, color=colors_bar, alpha=0.85,
               edgecolor='white')
ax.axvline(x=70, color=CORAL, linewidth=2, linestyle='--', alpha=0.7)
ax.axvline(x=30, color=GREEN, linewidth=2, linestyle='--', alpha=0.7)
ax.axvline(x=50, color=GRAY, linewidth=1, linestyle=':', alpha=0.5)

# Value labels
for bar, val in zip(bars, current_rsi.values):
    ax.text(val + 1, bar.get_y() + bar.get_height()/2,
            f'{val:.0f}', va='center', fontsize=10, fontweight='bold')

ax.set_xlabel('RSI', fontsize=12)
ax.set_title('Current RSI Snapshot\n(Red = overbought, Green = oversold)', fontweight='bold')
ax.set_xlim(0, 100)

# --- Bottom Left: RSI for one example tech stock ---
ax = axes[1, 0]
example_tech = 'NVDA'
rsi_ex = rsi_data[example_tech].dropna()
price_ex = sector_close[example_tech].loc[rsi_ex.index]

# Twin axis: price on left, RSI on right
ax.plot(price_ex.index, price_ex, color=NAVY, linewidth=1.5, label=f'{example_tech} Price')
ax.set_ylabel(f'{example_tech} Price ($)', fontsize=12, color=NAVY)
ax.set_title(f'Deep Dive: {example_tech} — Price vs. RSI', fontweight='bold')

ax2 = ax.twinx()
ax2.plot(rsi_ex.index, rsi_ex, color=CORAL, linewidth=1.5, alpha=0.8, label='RSI')
ax2.axhline(y=70, color=CORAL, linewidth=1, linestyle='--', alpha=0.4)
ax2.axhline(y=30, color=GREEN, linewidth=1, linestyle='--', alpha=0.4)
ax2.set_ylabel('RSI', fontsize=12, color=CORAL)
ax2.set_ylim(10, 90)

# Shade overbought/oversold periods
ax2.fill_between(rsi_ex.index, 70, rsi_ex.where(rsi_ex > 70), alpha=0.2, color=CORAL)
ax2.fill_between(rsi_ex.index, 30, rsi_ex.where(rsi_ex < 30), alpha=0.2, color=GREEN)

# Combined legend
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, fontsize=10, loc='upper left')

# --- Bottom Right: RSI for one example durables stock ---
ax = axes[1, 1]
example_dur = 'WMT'
rsi_ex2 = rsi_data[example_dur].dropna()
price_ex2 = sector_close[example_dur].loc[rsi_ex2.index]

ax.plot(price_ex2.index, price_ex2, color=TEAL, linewidth=1.5, label=f'{example_dur} Price')
ax.set_ylabel(f'{example_dur} Price ($)', fontsize=12, color=TEAL)
ax.set_title(f'Deep Dive: {example_dur} — Price vs. RSI', fontweight='bold')

ax3 = ax.twinx()
ax3.plot(rsi_ex2.index, rsi_ex2, color=CORAL, linewidth=1.5, alpha=0.8, label='RSI')
ax3.axhline(y=70, color=CORAL, linewidth=1, linestyle='--', alpha=0.4)
ax3.axhline(y=30, color=GREEN, linewidth=1, linestyle='--', alpha=0.4)
ax3.set_ylabel('RSI', fontsize=12, color=CORAL)
ax3.set_ylim(10, 90)

ax3.fill_between(rsi_ex2.index, 70, rsi_ex2.where(rsi_ex2 > 70), alpha=0.2, color=CORAL)
ax3.fill_between(rsi_ex2.index, 30, rsi_ex2.where(rsi_ex2 < 30), alpha=0.2, color=GREEN)

lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax3.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, fontsize=10, loc='upper left')

plt.tight_layout()
plt.show()

print('Reading the charts:')
print('• Top-left: Sector RSI averages — the shaded bands show how much individual stocks vary')
print('• Top-right: Snapshot of where each stock sits RIGHT NOW on the RSI scale')
print('• Bottom: Price overlaid with RSI — notice how RSI extremes often (but not always!) ')
print('  correspond to local price peaks and troughs')
print('\nKey question: Does RSI hitting 30 reliably predict a bounce? (Hint: think about EMH)')

In [ ]:
# ============================================================
# RSI SUMMARY STATISTICS: Compare the two sectors
# ============================================================

print('RSI Summary Statistics')
print('=' * 65)
print(f'{"Stock":>6s}  {"Mean":>6s}  {"Std":>6s}  {"Min":>6s}  {"Max":>6s}  {"Days>70":>7s}  {"Days<30":>7s}')
print('-' * 65)

for group_name, tickers in [('TECH', tech_tickers), ('DURABLES', durables_tickers)]:
    print(f'  --- {group_name} ---')
    for t in tickers:
        rsi = rsi_data[t].dropna()
        overbought = (rsi > 70).sum()
        oversold = (rsi < 30).sum()
        print(f'{t:>6s}  {rsi.mean():>6.1f}  {rsi.std():>6.1f}  {rsi.min():>6.1f}  '
              f'{rsi.max():>6.1f}  {overbought:>7d}  {oversold:>7d}')

# Sector averages
tech_mean_rsi = rsi_data[tech_tickers].mean().mean()
dur_mean_rsi = rsi_data[durables_tickers].mean().mean()
print(f'\nSector Average RSI:')
print(f'  Tech/SaaS:         {tech_mean_rsi:.1f}')
print(f'  Consumer Durables: {dur_mean_rsi:.1f}')

---

## Part 4: PE Ratios — The SaaS Decline and the Rise of "Boring" Stocks

One of the big stories in markets over the past year has been a **valuation rotation**:

- **SaaS / high-growth tech** stocks have seen their PE ratios compress — investors are less willing to pay a premium for growth after interest rates rose
- **"Boring" consumer staples and durables** stocks have seen their PEs *increase* — investors rotated into safer, dividend-paying names

This is a classic **growth-to-value rotation**. Let's visualize it.

### Why PE Ratios Matter

The **Price-to-Earnings ratio** tells you how much investors pay per dollar of earnings:

$$PE = \frac{\text{Stock Price}}{\text{Earnings Per Share}}$$

- **High PE** → investors expect strong future growth (or the stock is overvalued)
- **Low PE** → investors expect slow growth (or the stock is undervalued)

In [ ]:
# ============================================================
# DOWNLOAD PE RATIOS AND KEY METRICS
# ============================================================

metrics = []

for ticker in all_sector_tickers:
    try:
        info = yf.Ticker(ticker).info
        pe = info.get('trailingPE', None)
        fwd_pe = info.get('forwardPE', None)
        div_yield = info.get('dividendYield', 0)
        mkt_cap = info.get('marketCap', None)
        sector = 'Tech/SaaS' if ticker in tech_tickers else 'Consumer Durables/Staples'
        
        metrics.append({
            'Ticker': ticker,
            'Sector': sector,
            'Trailing PE': pe,
            'Forward PE': fwd_pe,
            'Div Yield': div_yield if div_yield else 0,
            'Mkt Cap ($B)': mkt_cap / 1e9 if mkt_cap else None
        })
        pe_str = f'{pe:.1f}' if pe else 'N/A'
        fwd_str = f'{fwd_pe:.1f}' if fwd_pe else 'N/A'
        print(f'  {ticker:6s}: Trailing PE = {pe_str:>6s}, Forward PE = {fwd_str:>6s}')
    except Exception as e:
        print(f'  {ticker}: Error — {e}')

metrics_df = pd.DataFrame(metrics).dropna(subset=['Trailing PE'])
print(f'\nSuccessfully retrieved metrics for {len(metrics_df)} stocks')

In [ ]:
# ============================================================
# PE RATIO COMPARISON: Bar Chart
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# --- Left: Trailing PE ---
ax = axes[0]
colors_pe = [NAVY if t in tech_tickers else TEAL for t in metrics_df['Ticker']]
bars = ax.bar(metrics_df['Ticker'], metrics_df['Trailing PE'], color=colors_pe,
              alpha=0.85, edgecolor='white', linewidth=1.5)

for bar, pe_val in zip(bars, metrics_df['Trailing PE']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{pe_val:.1f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

tech_avg_pe = metrics_df[metrics_df['Sector'] == 'Tech/SaaS']['Trailing PE'].mean()
dur_avg_pe = metrics_df[metrics_df['Sector'] == 'Consumer Durables/Staples']['Trailing PE'].mean()
ax.axhline(y=tech_avg_pe, color=NAVY, linewidth=2, linestyle='--', alpha=0.5)
ax.axhline(y=dur_avg_pe, color=TEAL, linewidth=2, linestyle='--', alpha=0.5)

ax.set_ylabel('Trailing PE Ratio', fontsize=13)
ax.set_title('Trailing PE: Tech vs. Boring Stocks', fontweight='bold', fontsize=13)
ax.tick_params(axis='x', rotation=45)

# --- Right: Forward PE (expectations) ---
ax = axes[1]
fwd_df = metrics_df.dropna(subset=['Forward PE'])
colors_fwd = [NAVY if t in tech_tickers else TEAL for t in fwd_df['Ticker']]
bars = ax.bar(fwd_df['Ticker'], fwd_df['Forward PE'], color=colors_fwd,
              alpha=0.85, edgecolor='white', linewidth=1.5)

for bar, pe_val in zip(bars, fwd_df['Forward PE']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{pe_val:.1f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_ylabel('Forward PE Ratio', fontsize=13)
ax.set_title('Forward PE: What the Market Expects\n(Lower = less growth expected)',
             fontweight='bold', fontsize=13)
ax.tick_params(axis='x', rotation=45)

# Shared legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=NAVY, alpha=0.85, label='Tech/SaaS'),
                   Patch(facecolor=TEAL, alpha=0.85, label='Consumer Durables/Staples')]
fig.legend(handles=legend_elements, fontsize=12, loc='upper center',
           bbox_to_anchor=(0.5, 1.02), ncol=2)

plt.tight_layout()
plt.show()

print(f'Tech/SaaS average trailing PE: {tech_avg_pe:.1f}')
print(f'Consumer Durables avg trailing PE: {dur_avg_pe:.1f}')
print(f'Tech PE premium: {tech_avg_pe/dur_avg_pe:.1f}x')
print(f'\nKey question: Is the tech premium justified by faster earnings growth?')
print('Or have investors overpaid for growth that\'s decelerating?')

In [ ]:
# ============================================================
# PERFORMANCE COMPARISON: Who's winning the race?
# ============================================================

# Normalized performance over the past year
sector_norm = (sector_close / sector_close.iloc[0]) * 100

tech_basket = sector_norm[tech_tickers].mean(axis=1)
durables_basket = sector_norm[durables_tickers].mean(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Left: Basket performance ---
ax = axes[0]
ax.plot(tech_basket.index, tech_basket, color=NAVY, linewidth=2.5,
        label=f'Tech/SaaS Basket ({tech_basket.iloc[-1]:.0f})')
ax.plot(durables_basket.index, durables_basket, color=TEAL, linewidth=2.5,
        label=f'Durables Basket ({durables_basket.iloc[-1]:.0f})')
ax.axhline(y=100, color='black', linewidth=0.5, linestyle=':')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Normalized Price (100 = start)', fontsize=12)
ax.set_title('Sector Performance: 1-Year Comparison', fontweight='bold')
ax.legend(fontsize=11)

# --- Right: Individual stock returns ---
ax = axes[1]
total_returns = ((sector_close.iloc[-1] / sector_close.iloc[0]) - 1) * 100
total_returns = total_returns[all_sector_tickers].sort_values(ascending=True)
colors_ret = [NAVY if t in tech_tickers else TEAL for t in total_returns.index]
ax.barh(total_returns.index, total_returns.values, color=colors_ret, alpha=0.85,
        edgecolor='white')
ax.axvline(x=0, color='black', linewidth=0.5)
for i, (ticker, ret) in enumerate(total_returns.items()):
    ax.text(ret + 0.5 if ret > 0 else ret - 0.5, i,
            f'{ret:+.1f}%', va='center', fontsize=10, fontweight='bold',
            ha='left' if ret > 0 else 'right')
ax.set_xlabel('Total Return (%)', fontsize=12)
ax.set_title('Individual Stock Returns (1 Year)', fontweight='bold')

plt.tight_layout()
plt.show()

# Risk-adjusted comparison
tech_returns = sector_close[tech_tickers].pct_change().mean(axis=1).dropna()
dur_returns = sector_close[durables_tickers].pct_change().mean(axis=1).dropna()

tech_sharpe = (tech_returns.mean() / tech_returns.std()) * np.sqrt(252)
dur_sharpe = (dur_returns.mean() / dur_returns.std()) * np.sqrt(252)

print(f'\nRisk-Adjusted Performance (Sharpe Ratio):')
print(f'  Tech/SaaS:         {tech_sharpe:.2f}')
print(f'  Consumer Durables: {dur_sharpe:.2f}')
print(f'\nHigher Sharpe = better return per unit of risk.')
print('Are the "boring" stocks actually the better investment on a risk-adjusted basis?')

### Connecting PE and Performance

There's a classic debate in finance: **do high-PE (growth) stocks outperform low-PE (value) stocks?**

- The **growth investor** says: you pay more for faster-growing companies, and the growth justifies the premium
- The **value investor** says: high-PE stocks are overpriced and eventually revert; low-PE stocks are bargains

The data over the past year has been interesting for this debate — let's visualize the PE vs. return relationship.

In [ ]:
# ============================================================
# PE vs. RETURN SCATTER PLOT (with separate sector regressions)
# ============================================================

# Merge PE data with returns
pe_return_df = metrics_df.copy()
total_ret_pct = ((sector_close.iloc[-1] / sector_close.iloc[0]) - 1) * 100
pe_return_df['1Y Return (%)'] = pe_return_df['Ticker'].map(
    lambda t: total_ret_pct[t] if t in total_ret_pct.index else np.nan
)
pe_return_df = pe_return_df.dropna(subset=['1Y Return (%)', 'Trailing PE'])

fig, ax = plt.subplots(figsize=(12, 8))

# Plot points by sector
for _, row in pe_return_df.iterrows():
    color = NAVY if row['Ticker'] in tech_tickers else TEAL
    marker = 's' if row['Ticker'] in tech_tickers else 'o'
    ax.scatter(row['Trailing PE'], row['1Y Return (%)'], s=180, color=color,
               edgecolor='black', zorder=5, marker=marker)
    ax.annotate(row['Ticker'], (row['Trailing PE'], row['1Y Return (%)']),
                textcoords='offset points', xytext=(10, 5), fontsize=11, fontweight='bold')

# --- Separate regression lines for each sector ---
tech_df = pe_return_df[pe_return_df['Sector'] == 'Tech/SaaS']
dur_df = pe_return_df[pe_return_df['Sector'] == 'Consumer Durables/Staples']

# Tech regression
if len(tech_df) >= 3:
    x_tech = tech_df['Trailing PE'].values
    y_tech = tech_df['1Y Return (%)'].values
    slope_t, intercept_t, r_t, p_t, _ = stats.linregress(x_tech, y_tech)
    x_line_t = np.linspace(x_tech.min() - 3, x_tech.max() + 3, 100)
    ax.plot(x_line_t, intercept_t + slope_t * x_line_t, color=NAVY, linewidth=2,
            linestyle='--', alpha=0.6, label=f'Tech trend (R²={r_t**2:.3f})')

# Durables regression
if len(dur_df) >= 3:
    x_dur = dur_df['Trailing PE'].values
    y_dur = dur_df['1Y Return (%)'].values
    slope_d, intercept_d, r_d, p_d, _ = stats.linregress(x_dur, y_dur)
    x_line_d = np.linspace(x_dur.min() - 3, x_dur.max() + 3, 100)
    ax.plot(x_line_d, intercept_d + slope_d * x_line_d, color=TEAL, linewidth=2,
            linestyle='--', alpha=0.6, label=f'Durables trend (R²={r_d**2:.3f})')

ax.axhline(y=0, color='gray', linewidth=0.5)
ax.set_xlabel('Trailing PE Ratio', fontsize=13)
ax.set_ylabel('1-Year Total Return (%)', fontsize=13)
ax.set_title('PE Ratio vs. Stock Performance — By Sector\n'
             'Does paying a higher PE within each sector pay off?',
             fontweight='bold', fontsize=14)

# Legend with sector markers + regression lines
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='s', color='w', markerfacecolor=NAVY, markersize=12,
           markeredgecolor='black', label='Tech/SaaS'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor=TEAL, markersize=12,
           markeredgecolor='black', label='Consumer Durables/Staples'),
]
if len(tech_df) >= 3:
    legend_elements.append(Line2D([0], [0], color=NAVY, linestyle='--', linewidth=2,
                                   label=f'Tech trend (R²={r_t**2:.3f})'))
if len(dur_df) >= 3:
    legend_elements.append(Line2D([0], [0], color=TEAL, linestyle='--', linewidth=2,
                                   label=f'Durables trend (R²={r_d**2:.3f})'))
ax.legend(handles=legend_elements, fontsize=11, loc='best')

plt.tight_layout()
plt.show()

# Print interpretation
print('Within-sector PE vs. Return:')
if len(tech_df) >= 3:
    direction_t = 'higher' if slope_t > 0 else 'lower'
    print(f'  Tech/SaaS:   slope = {slope_t:+.2f} (higher PE → {direction_t} returns, R² = {r_t**2:.3f})')
if len(dur_df) >= 3:
    direction_d = 'higher' if slope_d > 0 else 'lower'
    print(f'  Durables:    slope = {slope_d:+.2f} (higher PE → {direction_d} returns, R² = {r_d**2:.3f})')
print(f'\nThe two sectors live in different PE universes — that\'s why separate lines matter.')
print(f'A PE of 30 means very different things for a SaaS company vs. a cereal company.')

---

## Part 5: Putting It All Together — Reviewing Key Concepts

Let's tie together the key concepts from Weeks 1–3 using the data we've analyzed today.

In [ ]:
# ============================================================
# CONCEPT REVIEW: Volatility, Beta, and the Tariff Event
# ============================================================

# Calculate betas for our tariff stocks against SPY
# Use the full available period
spy_full = yf.download('SPY', period='1y')
spy_full.columns = spy_full.columns.get_level_values(0)
spy_ret_full = spy_full['Close'].pct_change().dropna()

print('Stock Characteristics: Tariff Event Stocks')
print('=' * 80)
print(f'{"Stock":>6s}  {"Group":>10s}  {"Beta":>6s}  {"Ann Vol":>8s}  {"Sharpe":>7s}  {"1Y Return":>10s}')
print('-' * 80)

tariff_all = tariff_sensitive + domestic_focused
tariff_data = yf.download(tariff_all, period='1y')['Close']
tariff_returns = tariff_data.pct_change().dropna()

for group_name, tickers in [('Tariff', tariff_sensitive), ('Domestic', domestic_focused)]:
    for ticker in tickers:
        try:
            stock_ret = tariff_returns[ticker].dropna()
            # Align with SPY
            common_idx = stock_ret.index.intersection(spy_ret_full.index)
            sr = stock_ret.loc[common_idx]
            mr = spy_ret_full.loc[common_idx]
            
            slope, _, _, _, _ = stats.linregress(mr, sr)
            ann_vol = sr.std() * np.sqrt(252)
            ann_ret = sr.mean() * 252
            sharpe = ann_ret / ann_vol if ann_vol > 0 else 0
            total_ret = (tariff_data[ticker].iloc[-1] / tariff_data[ticker].iloc[0] - 1) * 100
            
            print(f'{ticker:>6s}  {group_name:>10s}  {slope:>6.2f}  '
                  f'{ann_vol:>7.1%}  {sharpe:>7.2f}  {total_ret:>+9.1f}%')
        except Exception as e:
            print(f'{ticker:>6s}  {group_name:>10s}  Error: {e}')

print('\n\nKey connections to lecture material:')
print('• Beta measures sensitivity to the market — higher beta = bigger moves on tariff news')
print('• Annualized volatility captures total risk (both market + idiosyncratic)')
print('• Sharpe ratio tells you if higher returns are compensating for higher risk')
print('• CAPM (Week 2) predicts that higher-beta stocks should have higher expected returns')

---

## Part 6: Your Turn — Exercises

Now apply what you've learned. These exercises reinforce concepts from Weeks 1–3.

### Exercise 1: Your Own Event Study

Pick a different news event from the past year and conduct a mini event study. Some ideas:
- **A major earnings surprise** (e.g., NVDA, META, or TSLA earnings)
- **An FDA drug approval** (biotech stocks)
- **A Fed rate decision** (interest-rate sensitive stocks like banks or REITs)

Steps:
1. Identify the event date and 2–3 affected stocks
2. Download data for a window around the event
3. Calculate returns on the event day vs. normal days
4. (Bonus) Compute abnormal returns using CAPM

In [ ]:
# YOUR CODE HERE
# 1. Choose your event and affected tickers
# 2. Download data: yf.download(tickers, start='...', end='...')
# 3. Calculate returns: close.pct_change()
# 4. Print and visualize the event-day returns
# 5. Compare to normal daily returns (mean and std)
# Hint: A return that's >2 standard deviations from the mean is "abnormal"


### Exercise 2: RSI as a Trading Signal

Test whether RSI actually works as a **trading signal**. Strategy:
- **Buy** when RSI drops below 30 (oversold)
- **Sell** when RSI rises above 70 (overbought)
- Otherwise, hold

Implement this for one stock and compare cumulative returns to buy-and-hold.

**Think about:** This connects to Week 3's lesson on overfitting. Does a backtested RSI strategy actually beat buy-and-hold out of sample?

In [ ]:
# YOUR CODE HERE
# Hint: 
# ticker = 'AAPL'  # or any stock
# prices = sector_close[ticker]
# rsi = compute_rsi(prices)
# daily_return = prices.pct_change()
#
# # Create signal: 1 = long (invested), 0 = out of market
# signal = pd.Series(0, index=prices.index)
# position = 0  # 0 = out, 1 = in
# for i in range(1, len(rsi)):
#     if rsi.iloc[i] < 30:     # Buy signal
#         position = 1
#     elif rsi.iloc[i] > 70:   # Sell signal
#         position = 0
#     signal.iloc[i] = position
#
# # Strategy return = signal (shifted by 1 day) × next day's return
# strategy_return = signal.shift(1) * daily_return
#
# # Plot cumulative returns: strategy vs. buy-and-hold
# cum_strategy = (1 + strategy_return).cumprod()
# cum_bh = (1 + daily_return).cumprod()


### Exercise 3: Build a Sector Dashboard

Create a summary dashboard that compares the two sectors across multiple dimensions. For each sector, compute and display:

1. Average PE ratio
2. Average RSI (current)
3. Average 1-year return
4. Average Sharpe ratio
5. Average beta (vs. SPY)

Present the results in a clean table or bar chart.

In [ ]:
# YOUR CODE HERE
# Hint: Combine the calculations from earlier parts into one summary
# Use pd.DataFrame to create a comparison table
# For beta: use stats.linregress against SPY returns
# For Sharpe: (mean_return * 252) / (std_return * sqrt(252))


### Exercise 4: Moving Average Crossover on Tariff Stocks

From Week 1, you learned about **moving average crossovers** (Golden Cross / Death Cross). 

Pick one tariff-sensitive stock (e.g., GM or CAT) and:
1. Calculate 20-day and 50-day moving averages
2. Plot the price with both MAs
3. Identify any crossovers around the tariff event dates
4. Did the moving averages signal the crash and recovery?

**Think about:** Moving averages are lagging indicators. How does this relate to the speed of market adjustment you saw in Part 2?

In [ ]:
# YOUR CODE HERE
# 1. Pick a tariff-sensitive stock
# 2. Download data: yf.download(ticker, period='6mo')
# 3. Calculate MAs: close.rolling(20).mean(), close.rolling(50).mean()
# 4. Plot price + both MAs
# 5. Mark the tariff event dates on the chart
# 6. Check: Did the MA crossover signal the tariff-driven moves?


### Exercise 5: Return Distribution Around the Tariff Event

From Week 2, you learned that returns have **fat tails** — extreme events are more common than a normal distribution predicts.

For a tariff-sensitive stock:
1. Plot the histogram of daily returns over the past year
2. Overlay the normal distribution
3. Mark the tariff-event-day returns on the histogram
4. How many standard deviations was the tariff shock?

**Think about:** Was the tariff shock a "tail event"? How often should we expect moves this large if returns were truly normal?

In [ ]:
# YOUR CODE HERE
# Hint:
# ticker = 'GM'
# stock = yf.download(ticker, period='1y')
# stock.columns = stock.columns.get_level_values(0)
# stock['Return'] = stock['Close'].pct_change()
#
# # Histogram
# mu = stock['Return'].mean()
# sigma = stock['Return'].std()
# plt.hist(stock['Return'].dropna(), bins=50, density=True, alpha=0.7)
#
# # Normal curve
# x = np.linspace(mu - 4*sigma, mu + 4*sigma, 200)
# plt.plot(x, stats.norm.pdf(x, mu, sigma), 'r-', linewidth=2)
#
# # Mark event day return
# event_ret = stock.loc['2025-02-03', 'Return']
# n_sigma = (event_ret - mu) / sigma
# plt.axvline(x=event_ret, color='red', linewidth=2)


---

## Part 7: Prompt Engineering — News-Driven Analysis

Now practice using AI to assist with financial analysis. Write prompts that connect your empirical findings to the concepts from Weeks 1–3.

### Exercise 6: Explain Your Event Study Results

Write a prompt for an AI assistant that asks it to analyze the tariff event from a market efficiency perspective. Your prompt should:

- Include the specific abnormal return numbers from Part 2
- Reference how quickly prices adjusted (cumulative abnormal returns chart)
- Ask: Does the speed of adjustment support or challenge the EMH?
- Ask: If you had read the tariff news 5 minutes after it was published, could you have profited?
- Specify your audience: "Explain to an MBA student in an AI in Finance course"

**Tips:** The best prompts include actual data ("GM dropped 4.2% while SPY only fell 1.1%") and ask specific questions tied to theory.

```
YOUR PROMPT HERE:



```

**AI Response:** (Paste the response here)

```



```

**Your evaluation:** Was the AI's analysis consistent with what you observed in the data?

### Exercise 7: Growth vs. Value — AI Debate

Write a prompt asking an AI to make the case **for** and **against** investing in high-PE tech stocks vs. low-PE consumer staples, using your actual PE ratios and return data from Part 4.

Ask the AI to address:
- How the SaaS valuation decline relates to rising interest rates
- Whether "boring" stocks with rising PEs represent good value or a crowded trade
- What the Sharpe ratios tell you about which sector has been a better investment recently
- How this connects to the concepts of beta, CAPM, and systematic vs. idiosyncratic risk

```
YOUR PROMPT HERE:



```

---

## Part 8: Reflection

Answer these questions based on your work today.

### 1. Event Studies and Market Efficiency

Based on the tariff event analysis, do you think markets reacted efficiently? Was there an opportunity to profit after the news was public?

*Your answer:*


### 2. RSI: Useful Signal or Noise?

After comparing RSI across tech and consumer durables, do you think RSI provides actionable trading signals? How does this connect to the Week 3 lesson about technical indicators and EMH?

*Your answer:*


### 3. The Growth-Value Rotation

Why might "boring" consumer staples stocks be outperforming high-PE tech stocks on a risk-adjusted basis? What does this tell you about market efficiency and investor behavior?

*Your answer:*


### 4. Connecting the Dots

This lab used concepts from all three weeks: returns and Sharpe ratios (Week 1), CAPM betas and abnormal returns (Week 2), and technical indicators/market efficiency (Week 3). Which concept was most useful for understanding the tariff event? Which concept do you still find confusing?

*Your answer:*


---

## Key Takeaways

1. **Event studies** are a core tool in finance — they isolate the effect of news by measuring abnormal returns beyond what the market (beta) predicted
2. **The tariff reversal** provides a clean natural experiment: tariff-sensitive stocks moved, domestic stocks didn't, and the reversal was quick — consistent with market efficiency
3. **Options (puts and calls)** let you trade news events with defined risk — but option prices already reflect the market's estimate of event probabilities (implied volatility)
4. **RSI** is a widely-used momentum indicator, but its predictive power is limited — consistent with the EMH and Week 3's findings about technical indicators
5. **PE ratios** differ systematically across sectors. Tech stocks carry a growth premium, but that premium has been shrinking as SaaS growth decelerates
6. **"Boring" stocks** have seen PE expansion and strong risk-adjusted returns — a reminder that Sharpe ratio, not raw returns, is the right way to compare investments
7. **Market efficiency** doesn't mean prices are always right — it means prices adjust quickly to new information, making it hard to profit from public news
8. **Everything connects**: returns (Week 1) → risk and CAPM (Week 2) → efficiency and prediction (Week 3). This lab applied all three to a real-world event.

---

*Week 3 Practice Lab — BUS 696: Generative AI in Finance*